# 02 – Preprocessing

Steps covered:
1. Feature engineering – 6 Keplerian elements + 2 sinusoidal time encodings
2. Sliding-window sequence construction
3. Min-Max normalisation
4. Chronological 80/20 train/test split

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import (
    load_tle_file, build_dataset, create_windows, split_and_normalize,
    WINDOW_SIZE, STEP_MIN, HOURS
)

## 2.1 Build dataset from all TLEs

In [ ]:
tle_list = load_tle_file('../data/starlink_tle.txt')
records  = build_dataset(tle_list, hours=HOURS, step_min=STEP_MIN)
print(f'Total records: {len(records)}')
print(f'Feature vector shape: {records[0]["features"].shape}  (8-dim)')

## 2.2 Feature descriptions

$$\mathbf{e}_k = [i,\; \Omega,\; e,\; \omega,\; M,\; n,\; \sin(2\pi t_k/T_\oplus),\; \cos(2\pi t_k/T_\oplus)]^\top$$

In [ ]:
features = np.array([r['features'] for r in records])
names    = ['inclination', 'RAAN', 'eccentricity', 'arg_perigee',
            'mean_anomaly', 'mean_motion', 'sin_t', 'cos_t']

fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
for ax, col, nm in zip(axes.flat, range(8), names):
    ax.plot(features[:576, col], linewidth=0.8)
    ax.set_title(nm)
    ax.grid(linestyle='--', alpha=0.4)
plt.suptitle('Feature time-series (first satellite, 48 h)')
plt.tight_layout()
plt.show()

## 2.3 Sliding-window sequences

Window size $W = 12$ steps (1 hour):
$$\mathbf{X}_k = [\mathbf{e}_{k-11}, \dots, \mathbf{e}_k] \in \mathbb{R}^{12 \times 8}, \quad \mathbf{y}_k = \mathbf{r}_{k+1}$$

In [ ]:
X, y = create_windows(records, window_size=WINDOW_SIZE)
print(f'X shape : {X.shape}   # (num_windows, window_size, features)')
print(f'y shape : {y.shape}   # (num_windows, 3)')

## 2.4 Min-Max normalisation & chronological split

$$\tilde{f} = \frac{f - f_{\min}}{f_{\max} - f_{\min}}$$

In [ ]:
X_train, y_train, X_test, y_test, x_sc, y_sc = split_and_normalize(X, y)
print(f'X_train : {X_train.shape}  (min={X_train.min():.3f}, max={X_train.max():.3f})')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}  (normalised, min={y_train.min():.3f})')
print(f'y_test  : {y_test.shape}')

## 2.5 Verify no data leakage

Training indices are strictly before test indices (no shuffle).

In [ ]:
split = int(len(X) * 0.8)
print(f'Train covers timesteps 0 … {split - 1}')
print(f'Test  covers timesteps {split} … {len(X) - 1}')
print('No overlap ✓' if split <= len(X) else 'OVERLAP detected!')